In [15]:
!pip install ninja

In [16]:
!cd kernelforge/ && git pull origin main


remote: Enumerating objects: 11, done.
remote: Counting objects: 100% (11/11), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 6 (delta 3), reused 6 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (6/6), 948 bytes | 316.00 KiB/s, done.
From https://github.com/marcalph/kernelforge
 * branch            main       -> FETCH_HEAD
   d685174..3cb1bf0  main       -> origin/main
Updating d685174..3cb1bf0
Fast-forward
 src/grayscale.ipynb      | 23 +++++++++++------------
 src/kernels/grayscale.cu | 22 +++++++---------------
 2 files changed, 18 insertions(+), 27 deletions(-)


In [20]:
# Run this cell if load_inline fails with "cannot open shared object file"
# (happens when a previous compilation failed and left a broken cache entry)
!rm -rf /root/.cache/torch_extensions/py312_cu128/rgb_to_grayscale_extension/


In [22]:
# Run this cell if load_inline fails with "cannot open shared object file"
# (happens when a previous compilation failed and left a broken cache entry)
!ls /root/.cache/torch_extensions/py312_cu128


In [23]:
from pathlib import Path
import torch
from torchvision.io import read_image, write_png
from torch.utils.cpp_extension import load_inline


def compile_extension():
    cuda_source = Path("kernelforge/src/kernels/grayscale.cu").read_text()
    cpp_source = "torch::Tensor rgb_to_grayscale(torch::Tensor image);"

    return load_inline(
        name="rgb_to_grayscale_extension",
        cpp_sources=cpp_source,
        cuda_sources=cuda_source,
        functions=["rgb_to_grayscale"],
        with_cuda=True,
        extra_cuda_cflags=["-O2"],
        verbose=True,
        build_directory='./cuda_build',
    )


def main():
    """
    Read input image, convert it to grayscale via custom CUDA kernel and write out as png.
    """
    ext = compile_extension()

    x = read_image("kernelforge/src/lenna.jpg").permute(1, 2, 0).cuda()
    print("Input image:", x.shape, x.dtype, "mean:", x.float().mean().item())

    assert x.dtype == torch.uint8

    y = ext.rgb_to_grayscale(x)

    print("Output image:", y.shape, y.dtype, "mean:", y.float().mean().item())
    write_png(y.permute(2, 0, 1).cpu(), "output.png")


main()


FileNotFoundError: [Errno 2] No such file or directory: './cuda_build/main.cpp'